<a href="https://colab.research.google.com/github/tarannump096-cpu/NLP/blob/main/Santander_Customer_Transaction_Prediction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
df_train=pd.read_csv("/content/train.csv")

In [ ]:
df_train.columns

Index(['ID_code', 'target', 'var_0', 'var_1', 'var_2', 'var_3', 'var_4',
       'var_5', 'var_6', 'var_7',
       ...
       'var_190', 'var_191', 'var_192', 'var_193', 'var_194', 'var_195',
       'var_196', 'var_197', 'var_198', 'var_199'],
      dtype='object', length=202)

In [ ]:
df_test=pd.read_csv("/content/test.csv")

In [ ]:
df_test.columns

Index(['ID_code', 'var_0', 'var_1', 'var_2', 'var_3', 'var_4', 'var_5',
       'var_6', 'var_7', 'var_8',
       ...
       'var_190', 'var_191', 'var_192', 'var_193', 'var_194', 'var_195',
       'var_196', 'var_197', 'var_198', 'var_199'],
      dtype='object', length=201)

In [ ]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 695 entries, 0 to 694
Columns: 202 entries, ID_code to var_199
dtypes: float64(200), int64(1), object(1)
memory usage: 1.1+ MB


In [ ]:
df_train.drop(columns=["ID_code"],inplace=True)

In [ ]:
df_test.drop(columns=["ID_code"],inplace=True)

In [ ]:
def adding_rows_to_x(df):
    df_new=df.copy()
    df_new["row_mean"]=df_new.mean(axis=1)
    df_new["row_sd"]=df_new.std(axis=1)
    df_new["row_median"]=df_new.median(axis=1)
    df_new["row_min"]=df_new.min(axis=1)
    df_new["row_max"]=df_new.max(axis=1)
    df_new["row_sum"]=df_new.sum(axis=1)
    df_new["row_skew"]=df_new.skew(axis=1)
    df_new["pos_cnt"]=(df_new>0).sum(axis=1)
    df_new["neg_cnt"]=(df_new<0).sum(axis=1)
    return df_new

In [ ]:
x=df_train.drop(columns=["target"])
y=df_train["target"]

In [ ]:
X=adding_rows_to_x(x)

In [ ]:
X.shape

(695, 209)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
xtrain,xtest,ytrain,ytest=train_test_split(X,y,test_size=0.2,random_state=365,stratify=y)

In [ ]:
df_train["target"].value_counts()

,count
target,
0,629
1,66


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [ ]:
ss=StandardScaler()

In [ ]:
xtrain_scaled=ss.fit_transform(xtrain)

In [ ]:
xtest_scale=ss.transform(xtest)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model1 = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),  # Impute missing values with the mean
    ('scaler', StandardScaler()),                 # Scale features
    ('classifier', LogisticRegression(random_state=365))  # Logistic Regression model
])

In [ ]:
params_grid=[
    {
        "classifier__penalty":["l2"],
        "classifier__C":[1.0,2.0,0.1,0.01],
        "classifier__solver":["lbfgs"],
        "classifier__class_weight":["balanced"]
    },
    {
        "classifier__penalty":["l1", "l2"],
        "classifier__C":[1.0,2.0,0.1,0.01],
        "classifier__solver":["liblinear"],
        "classifier__class_weight":["balanced"]
    }
]

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
grid=GridSearchCV(
    estimator=model1, # Use the pipeline as the estimator
    param_grid=params_grid,
    cv=5,
    scoring="accuracy",
    error_score='raise' # Raise errors explicitly for better debugging
)

In [ ]:
grid.fit(xtrain,ytrain) # Pass original xtrain, pipeline handles imputation and scaling

GridSearchCV(cv=5, error_score='raise',
             estimator=Pipeline(steps=[('imputer', SimpleImputer()),
                                       ('scaler', StandardScaler()),
                                       ('classifier',
                                        LogisticRegression(random_state=365))]),
             param_grid=[{'classifier__C': [1.0, 2.0, 0.1, 0.01],
                          'classifier__class_weight': ['balanced'],
                          'classifier__penalty': ['l2'],
                          'classifier__solver': ['lbfgs']},
                         {'classifier__C': [1.0, 2.0, 0.1, 0.01],
                          'classifier__class_weight': ['balanced'],
                          'classifier__penalty': ['l1', 'l2'],
                          'classifier__solver': ['liblinear']}],
             scoring='accuracy')

In [ ]:
grid.best_params_

{'classifier__C': 0.01,
 'classifier__class_weight': 'balanced',
 'classifier__penalty': 'l1',
 'classifier__solver': 'liblinear'}

In [ ]:
grid.best_score_

np.float64(0.9046814671814672)

In [ ]:
# Initialize imputer and scaler for lr_model
imputer_for_lr = SimpleImputer(strategy='mean')
scaler_for_lr = StandardScaler()

# Impute and scale xtrain for lr_model
xtrain_imputed_for_lr = imputer_for_lr.fit_transform(xtrain)
xtrain_processed_for_lr = scaler_for_lr.fit_transform(xtrain_imputed_for_lr)

lr_model = LogisticRegression(C=2.0, class_weight='balanced', penalty='l2', solver='lbfgs', random_state=365)

In [ ]:
lr_model.fit(xtrain_processed_for_lr,ytrain)

LogisticRegression(C=2.0, class_weight='balanced', random_state=365)

In [ ]:
ypred = lr_model.predict(xtest_scale)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(ytest,ypred)

0.8633093525179856

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(ytest,ypred))

              precision    recall  f1-score   support

           0       0.91      0.94      0.93       126
           1       0.12      0.08      0.10        13

    accuracy                           0.86       139
   macro avg       0.52      0.51      0.51       139
weighted avg       0.84      0.86      0.85       139



In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(ytest,ypred)

array([[119,   7],
       [ 12,   1]])

In [ ]:
original_df_test = pd.read_csv("/content/test.csv")
sample_sub = pd.DataFrame({'ID_code': original_df_test['ID_code']})

# Ensure the test data used for prediction has the same number of rows as original_df_test
current_df_test_for_pred = original_df_test.drop(columns=["ID_code"])

X_test_features = adding_rows_to_x(current_df_test_for_pred)
X_test_imputed = imputer_for_lr.transform(X_test_features)
X_test_processed = scaler_for_lr.transform(X_test_imputed)

test_pred = lr_model.predict(X_test_processed)

sample_sub['target'] = test_pred
sample_sub.to_csv("/content/sample_submission.csv", index=False)

print("Submission file created ✅")

Submission file created ✅
